# PTU Sizing Demo Notebook

This notebook packages the indicative PTU workshop/demo logic into a reusable standalone artifact. It estimates baseline PTUs, compares PTU vs PAYGO monthly cost assumptions, and returns an architecture suggestion.

**Important:** This is not the official PTU calculator. Replace model throughput, minimum commit, and pricing assumptions with validated values before customer use.


In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Single source of truth shared with the Streamlit app so the two cannot drift.
from ptu_core import (
    DEFAULTS,
    DEPLOYMENT_TYPES,
    MODEL_PRESETS,
    available_deployment_types,
    deployment_minimums,
    calculate,
)

DEFAULTS


In [ ]:
def build_inputs(model_preset="gpt-4.1", deployment_type="Global", **overrides):
    """Build a calculate() input dict from a model preset + deployment type.

    The preset fills model throughput and output weighting; the deployment type
    selects the minimum PTU commit and scale increment (Global/Data Zone use the
    lower minimums, Regional uses the larger model-specific ones). Any keyword
    override (e.g. avg_rpm=120) wins over the preset/defaults.
    """
    preset = MODEL_PRESETS.get(model_preset, {})
    allowed = available_deployment_types(preset)
    if deployment_type not in allowed:
        raise ValueError(
            f"{model_preset} does not support {deployment_type} provisioned. "
            f"Available: {', '.join(allowed)}."
        )
    min_ptu, increment = deployment_minimums(preset, deployment_type)

    values = DEFAULTS.copy()
    values.update({k: v for k, v in preset.items() if k in DEFAULTS})
    values["min_ptu_commit"] = min_ptu
    values["ptu_scale_increment"] = increment
    values.update(overrides)
    return values

# Show which deployment types each model preset supports.
pd.DataFrame(
    [(name, ", ".join(available_deployment_types(p))) for name, p in MODEL_PRESETS.items()],
    columns=["Model preset", "Available deployment types"],
)


## Edit inputs and run

Update any values below for a customer scenario, then run the next cells.


In [ ]:
# Choose a model preset and deployment type, then override workload values.
MODEL_PRESET = "gpt-4.1"          # e.g. gpt-5.1, gpt-5-mini, gpt-4.1-mini, gpt-4o, Llama-3.3-70B
DEPLOYMENT_TYPE = "Global"        # Global, Data Zone, or Regional (availability varies by model)

inputs = build_inputs(
    MODEL_PRESET,
    DEPLOYMENT_TYPE,
    avg_rpm=60,
    avg_input_tokens=1800,
    avg_output_tokens=650,
    p95_multiplier=1.8,
)
inputs


In [ ]:
result = calculate(inputs)
summary = pd.DataFrame([
    ('Recommended PTUs', result['recommended_ptu']),
    ('Peak reference PTUs', result['peak_reference_ptu']),
    ('Baseline input-equivalent TPM', result['baseline_tpm']),
    ('P95 input-equivalent TPM', result['p95_tpm']),
    ('PTU monthly', result['ptu_monthly']),
    ('PAYGO monthly', result['paygo_monthly']),
    ('Burst ratio', result['burst_ratio']),
], columns=['Metric', 'Value'])
summary


In [ ]:
display(Markdown(f"### {result['architecture']['label']}"))
display(Markdown(result['architecture']['summary']))
display(Markdown(f"**Why:** {result['architecture']['reason']}"))
display(Markdown(f"**Reservation note:** {result['reservation_note']}"))


In [ ]:
chart_df = pd.DataFrame({
    'Scenario': ['Baseline PTU', 'Peak ref PTU'],
    'PTUs': [result['recommended_ptu'], result['peak_reference_ptu']]
})
ax = chart_df.plot(kind='bar', x='Scenario', y='PTUs', legend=False, figsize=(8, 4), color=['#2563EB', '#94A3B8'])
ax.set_title('Baseline vs peak PTU view')
ax.set_ylabel('PTUs')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Optional widget UI

If `ipywidgets` is available, run the cell below for a lightweight interactive notebook experience.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    model_w = widgets.Dropdown(options=list(MODEL_PRESETS.keys()), value='gpt-4.1', description='Model')
    deploy_w = widgets.Dropdown(options=available_deployment_types(MODEL_PRESETS['gpt-4.1']), description='Deploy')
    avg_rpm_w = widgets.FloatText(value=DEFAULTS['avg_rpm'], description='Avg RPM')
    in_w = widgets.FloatText(value=DEFAULTS['avg_input_tokens'], description='In tokens')
    out_w = widgets.FloatText(value=DEFAULTS['avg_output_tokens'], description='Out tokens')
    p95_w = widgets.FloatSlider(value=DEFAULTS['p95_multiplier'], min=1, max=5, step=0.1, description='P95 x')
    cache_w = widgets.FloatSlider(value=DEFAULTS['cache_rate'], min=0, max=0.9, step=0.05, description='Cache')
    btn = widgets.Button(description='Recalculate', button_style='primary')
    out = widgets.Output()

    def on_model_change(_=None):
        # Restrict the deployment dropdown to the types the chosen model supports.
        options = available_deployment_types(MODEL_PRESETS[model_w.value])
        deploy_w.options = options
        if deploy_w.value not in options:
            deploy_w.value = options[0]

    def refresh(_=None):
        vals = build_inputs(
            model_w.value,
            deploy_w.value,
            avg_rpm=avg_rpm_w.value,
            avg_input_tokens=in_w.value,
            avg_output_tokens=out_w.value,
            p95_multiplier=p95_w.value,
            cache_rate=cache_w.value,
        )
        r = calculate(vals)
        with out:
            clear_output()
            display(pd.DataFrame([
                ('Model / deployment', f"{model_w.value} · {deploy_w.value}"),
                ('Min PTU commit / increment', f"{int(vals['min_ptu_commit'])} / {int(vals['ptu_scale_increment'])}"),
                ('Recommended PTUs', r['recommended_ptu']),
                ('PTU monthly (1-mo reserved)', round(r['ptu_monthly'], 2)),
                ('PAYGO monthly', round(r['paygo_monthly'], 2)),
                ('Architecture', r['architecture']['label']),
            ], columns=['Metric', 'Value']))

    model_w.observe(on_model_change, names='value')
    deploy_w.observe(refresh, names='value')
    btn.on_click(refresh)
    ui = widgets.VBox([
        widgets.HBox([model_w, deploy_w]),
        widgets.HBox([avg_rpm_w, in_w, out_w]),
        widgets.HBox([p95_w, cache_w]),
        btn,
        out,
    ])
    display(ui)
    refresh()
except Exception as exc:
    print('ipywidgets UI not available in this environment:', exc)
